# WordPiece from scratch — Solution

Reference implementation for `03_wordpiece_exercise.ipynb`.

## Setup

In [1]:
from collections import Counter

In [2]:
def pre_tokenize_wp(corpus):
    word_freqs = Counter()
    for sentence in corpus:
        for word in sentence.lower().split():
            chars = [word[0]] + ['##' + c for c in word[1:]]
            word_freqs[tuple(chars)] += 1
    return dict(word_freqs)


def get_pair_counts(word_freqs):
    pair_counts = Counter()
    for word, freq in word_freqs.items():
        for pair in zip(word, word[1:]):
            pair_counts[pair] += freq
    return pair_counts


def merge_token(a, b):
    return a + (b[2:] if b.startswith('##') else b)


def merge_pair(pair, word_freqs):
    updated = {}
    for word, freq in word_freqs.items():
        i = 0
        new_word = []
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(merge_token(word[i], word[i + 1]))
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        updated[tuple(new_word)] = freq
    return updated

## Solution 1 — PMI scoring

In [3]:
def get_token_counts(word_freqs):
    token_counts = Counter()
    for word, freq in word_freqs.items():
        for token in word:
            token_counts[token] += freq
    return token_counts


def get_pmi_scores(word_freqs):
    pair_counts  = get_pair_counts(word_freqs)
    token_counts = get_token_counts(word_freqs)
    pmi_scores   = {}
    for pair, freq in pair_counts.items():
        pmi_scores[pair] = freq / (token_counts[pair[0]] * token_counts[pair[1]])
    return pmi_scores

In [4]:
# Sanity check
corpus = ['the cat sat', 'the bat sat', 'queen quick']
word_freqs = pre_tokenize_wp(corpus)
scores = get_pmi_scores(word_freqs)
for pair, s in sorted(scores.items(), key=lambda x: -x[1])[:5]:
    print(pair, '->', round(s, 5))

('##i', '##c') -> 1.0
('##c', '##k') -> 1.0
('t', '##h') -> 0.5
('q', '##u') -> 0.5
('##u', '##i') -> 0.5


**Why PMI over raw frequency?**  BPE's frequency criterion can pick pairs that are common simply because both tokens are common (e.g., `('t','h')` because both letters are everywhere). PMI normalizes by the marginal probabilities — pairs whose co-occurrence is *surprising* given the marginals get higher scores. This produces tokens that capture genuine linguistic units like `'qu'` (almost never apart) over `'th'` (common but largely independent).

## Solution 2 — `WordPieceTokenizer`

The full tokenizer: `train` learns a vocab via PMI-driven merges, `encode` segments new text via greedy longest-match.

**Initial vocab seeding.** Real BERT WordPiece seeds *both* the bare form (e.g. `'a'`) and the continuation form (e.g. `'##a'`) for every character — so any character can appear at any position regardless of where it was first observed in the corpus. We do the same below.

In [5]:
class WordPieceTokenizer:
    def __init__(self):
        self.vocab = set()

    def train(self, corpus, vocab_size):
        word_freqs = pre_tokenize_wp(corpus)

        # Seed: bare AND '##' form for every character observed in the corpus.
        self.vocab = set()
        for word_tuple in word_freqs.keys():
            for piece in word_tuple:
                self.vocab.add(piece)
                if piece.startswith('##'):
                    self.vocab.add(piece[2:])
                else:
                    self.vocab.add('##' + piece)

        # Merge until target vocab size is reached.
        while len(self.vocab) < vocab_size:
            pmi_scores = get_pmi_scores(word_freqs)
            if not pmi_scores:
                break
            best_pair = max(pmi_scores, key=pmi_scores.get)
            word_freqs = merge_pair(best_pair, word_freqs)
            self.vocab.add(merge_token(best_pair[0], best_pair[1]))

    def encode(self, text):
        encoded = []
        for word in text.lower().split():
            pieces = []
            start = 0
            while start < len(word):
                end = len(word)
                found = None
                while start < end:
                    candidate = word[start:end]
                    if start > 0:
                        candidate = '##' + candidate
                    if candidate in self.vocab:
                        found = candidate
                        break
                    end -= 1
                if found is None:
                    pieces = ['UNK']
                    break
                pieces.append(found)
                start = end
            encoded.extend(pieces)
        return encoded

## Solution 3 — Greedy longest-match encoding

**Why greedy longest-match instead of replaying merges?** WordPiece's training algorithm doesn't produce a unique merge sequence — different orderings of equivalent PMI values can give different vocabularies. So WordPiece commits only to the final vocabulary, and uses a deterministic encoding rule (greedy longest-match) at inference. Trade-off: encoding can sometimes produce a different segmentation than what training would have produced for the same word.

**Why `##` for continuation?** It's a debugging affordance and a useful signal for downstream tasks like NER. `'token'` tells you 'this is the start of a word'; `'##ization'` tells you 'this is glued to whatever came before.' BPE uses a different convention (a leading space/`Ġ` for word starts), but it's the same idea expressed differently.

In [6]:
# Sanity check
tok = WordPieceTokenizer()
tok.train(['the cat sat', 'the dog ran', 'the cat sat'] * 10, vocab_size=32)
print('vocab:', sorted(tok.vocab))
print('encode("the cat"):  ', tok.encode('the cat'))
print('encode("thecatxyz"):', tok.encode('thecatxyz'))   # made-up word, should fragment or UNK

vocab: ['##a', '##c', '##d', '##e', '##g', '##h', '##n', '##o', '##r', '##s', '##t', 'a', 'c', 'ca', 'cat', 'd', 'do', 'dog', 'e', 'g', 'h', 'n', 'o', 'r', 'ra', 'ran', 's', 'sa', 'sat', 't', 'th', 'the']
encode("the cat"):   ['the', 'cat']
encode("thecatxyz"): ['UNK']


## Run the tests

In [7]:
from tests import run_wordpiece_tests
run_wordpiece_tests(WordPieceTokenizer, get_pmi_scores)

Running WordPiece...
  ✓ PMI prefers strong association over raw frequency
  ✓ train: vocab populated within target
  ✓ encode: returns list of strings
  ✓ encode: uses '##' for continuation pieces

  All 4 checks passed ✓



True